In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

**Loading and Preparing the dataset**

In [2]:
url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
data = pd.read_csv(url,usecols = ['engine_displacement','horsepower','vehicle_weight','model_year','fuel_efficiency_mpg'])
data.head()

,model_year,engine_displacement,horsepower,vehicle_weight,fuel_efficiency_mpg
0,2006,2180,243.0,3870,31.9
1,2008,2390,272.0,4210,31.3
2,1996,2320,267.0,4240,27.5
3,1989,2130,258.0,4490,28.5
4,1994,2580,304.0,4510,31.0


Question 1

In [3]:
missing = data.columns[data.isna().sum() > 0].to_list()
missing

['horsepower']

Question 2
* What's the median (50% percentile) for variable 'horsepower'?

In [4]:
data.horsepower.median()

np.float64(254.0)

In [5]:
n = len(data)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = data.iloc[idx[:n_train]]
df_val = data.iloc[idx[n_train:n_train + n_val]]
df_test = data.iloc[idx[n_train + n_val:]]

Question 3
* We need to deal with missing values for the column from Q1.
* We have two options: fill it with 0 or with the mean of this variable.
* Try both options. For each, train a linear regression model without regularization using the code from the lessons.
* For computing the mean, use the training only!
* Use the validation dataset to evaluate the models and compare the RMSE of each option.
* Round the RMSE scores to 3 decimal digits using round(score, 3). This keeps the imputation difference visible in this release.
* Which option gives better RMSE?

In [6]:
df_train_zero = df_train.copy()
df_val_zero = df_val.copy()

In [7]:
# Fillna with 0

df_train_zero['horsepower']  = df_train_zero['horsepower'].fillna(0)
df_val_zero['horsepower'] = df_val_zero['horsepower'].fillna(0)
df_train_zero['horsepower'].isna().any(),df_val_zero['horsepower'].isna().any()


(np.False_, np.False_)

Extract the feature and target columns

In [8]:
feature = ['model_year', 'engine_displacement', 'horsepower', 'vehicle_weight']
target = 'fuel_efficiency_mpg'

In [9]:
X_train_zero = df_train_zero[feature].values
y_train_zero = df_train_zero[target].values

X_val_zero = df_val_zero[feature].values
y_val_zero = df_val_zero[target].values

In [10]:
X_train_zero.shape

(6000, 4)

Defining the train function

In [11]:
def train_linear_reg(X,y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones,X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0],w_full[1:]

Prediction

In [12]:
w0,w_zero = train_linear_reg(X_train_zero,y_train_zero)
y_pred_zero = w0 + X_train_zero.dot(w_zero)
w0,w_zero

(np.float64(-153.8849618825958),
 array([ 1.02146785e-01, -1.51505520e-03, -4.74024796e-06, -3.95418397e-03]))

Evaluation using RMSE

In [13]:
def rmse (y,y_pred):
    se = (y-y_pred) ** 2
    mse = se.mean()
    return np.sqrt(mse)

In [14]:
rmse_zero = round(rmse(y_train_zero,y_pred_zero),3)
rmse_zero

np.float64(2.2)

Filling missing values with mean

In [15]:
df_train_mean = df_train.copy()
df_val_mean = df_val.copy()

In [16]:
hp_mean = df_train['horsepower'].mean()
df_train_mean['horsepower'] = df_train['horsepower'].fillna(hp_mean)
df_val_mean['horsepower'] = df_val['horsepower'].fillna(hp_mean)

Feature and target columns

In [17]:
X_train_mean = df_train_mean[feature].values
y_train_mean = df_train_mean[target].values

X_val_mean = df_val_mean[feature].values
y_val_mean = df_val_mean[target].values

Prediction

In [18]:
w0 ,w_mean = train_linear_reg(X_train_mean,y_train_mean)

y_pred_mean = w0 + X_train_mean.dot(w_mean)

In [19]:
rmse_mean = round(rmse(y_train_mean,y_pred_mean),3)
rmse_mean

np.float64(2.198)

Question 4
* Now let's train a regularized linear regression.
* For this question, fill the NAs with 0.
* Try different values of r from this list: [0, 0.01, 0.1, 1, 5, 10, 100].
* Use RMSE to evaluate the model on the validation dataset.
* Round the RMSE scores to 4 decimal digits. This keeps the small but real  regularization differences visible instead of turning several choices into a tie.
* Which r gives the best RMSE?

In [20]:
def train_linear_regular(X,y,r):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones,X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0],w_full[1:]

In [21]:
r = [0,0.01,0.1,1,5,10,100]
for i in r:
    w0, w = train_linear_regular(X_train_zero, y_train_zero, i)
    y_pred_reg = w0 + X_val_zero.dot(w)
    rmse_rg = rmse(y_val_zero, y_pred_reg)
    # Print the r value alongside its resulting RMSE
    print(f"r = {i:5} -> RMSE: {rmse_rg:.4f}")

r =     0 -> RMSE: 2.2053
r =  0.01 -> RMSE: 2.2058
r =   0.1 -> RMSE: 2.2241
r =     1 -> RMSE: 2.3492
r =     5 -> RMSE: 2.4094
r =    10 -> RMSE: 2.4195
r =   100 -> RMSE: 2.4292


Question 5
* We used seed 42 for splitting the data. Let's find out how selecting the seed influences our score.
* Try different seed values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9].
* For each seed, do the train/validation/test split with 60%/20%/20% distribution.
* Fill the missing values with 0 and train a model without regularization.
* For each seed, evaluate the model on the validation dataset and collect the RMSE scores.
* What's the standard deviation of all the scores? To compute the standard deviation, use np.std.
* Round the result to 3 decimal digits (round(std, 3))

**Function to prepare the feature datasets**

In [23]:
def prepare_X(df):
    df_num = df[feature]
    df_num = df_num.fillna(0)
    X = df_num.values
    return X

In [28]:
n = len(data)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

res = []
splits = {}
r = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
for i in r:

    # Set seed and shuffle indices
    np.random.seed(i)
    idx = np.arange(n)
    np.random.shuffle(idx)

    # Split the dataframes
    df_train = data.iloc[idx[:n_train]]
    df_val = data.iloc[idx[n_train:n_train + n_val]]
    df_test = data.iloc[idx[n_train + n_val:]]

    # Store train, val, and test inside the dictionary under the seed key
    splits[i] = {
        'train': df_train.copy(),
        'val': df_val.copy(),
        'test': df_test.copy()
    }

    # Prepare X and y for train and validation
    X_train = prepare_X(df_train)
    y_train = df_train[target].values

    X_val = prepare_X(df_val)
    y_val = df_val[target].values

    # Train the model
    w0,w = train_linear_reg(X_train,y_train)

    # Predict on the validation set
    y_pred = X_val.dot(w) + w0

    # Calculate RMSE
    score = rmse(y_val,y_pred)
    res.append(score)

    print(f"Seed {i:2} -> Validation RMSE: {score:.3f}")

print(f'std : {np.std(res):.3f}')

Seed  0 -> Validation RMSE: 2.239
Seed  1 -> Validation RMSE: 2.202
Seed  2 -> Validation RMSE: 2.163
Seed  3 -> Validation RMSE: 2.204
Seed  4 -> Validation RMSE: 2.202
Seed  5 -> Validation RMSE: 2.246
Seed  6 -> Validation RMSE: 2.270
Seed  7 -> Validation RMSE: 2.191
Seed  8 -> Validation RMSE: 2.217
Seed  9 -> Validation RMSE: 2.207
std : 0.029


Question 6
* Split the dataset like previously, use seed 9.
* Combine train and validation datasets.
* Fill the missing values with 0 and train a model with r=0.001.
* What's the RMSE on the test dataset?

In [32]:
df_train_val = pd.concat([splits[9]['train'] , splits[9]['val']])

In [36]:
X_train = prepare_X(df_train_val)
y_train = df_train_val[target].values

X_test = prepare_X(df_test)
y_test = df_test[target].values

In [37]:
w0 , w = train_linear_regular(X_train,y_train,r=0.001)
y_pred = w0 + X_test.dot(w)

score = rmse(y_test,y_pred)
print(f'Score : {score:.3f}')

Score : 2.236
